In [1]:
# Project 8 Walk Forward Testing
import pandas as pd
import numpy as np

prices = [
    100, 102, 104, 101, 98,
    105, 110, 108, 112, 115,
    109, 107, 111, 117, 120,
    118, 121, 125, 123, 128,
    130, 127, 132, 135, 138,
    136, 140, 145, 143, 148,
    150, 147, 152, 155, 153,
    158, 160, 157, 162, 165
]

df = pd.DataFrame({'Prices': prices})
test_result = []
train_end = 20

for i in range(0,3):
    result = []
    train_df = df.iloc[:train_end].copy()
    for fast in range(2,6):
        for slow in range(6,11):
            train_df['Fast_MA'] = train_df['Prices'].rolling(window=fast).mean()
            train_df['Slow_MA'] = train_df['Prices'].rolling(window=slow).mean()
            train_df['Return'] = train_df['Prices'].pct_change()
            train_df['Signal'] = np.where(train_df['Fast_MA'] > train_df['Slow_MA'] ,1,0 )
            train_df['Strategy_Return'] = train_df['Signal'].shift(1) * train_df['Return']
            sharpe= train_df['Strategy_Return'].mean()/train_df['Strategy_Return'].std()
            result.append([fast,slow,sharpe])
    
    train_df1 = pd.DataFrame(result,columns=['Fast','Slow','Sharpe'])
    train_df1.sort_values(by='Sharpe',ascending = False,inplace = True)
    best_fast = train_df1['Fast'].iloc[0]
    best_slow = train_df1['Slow'].iloc[0]
    combined_df = df.iloc[:train_end+5].copy()
    combined_df['Fast_MA'] = combined_df['Prices'].rolling(window = best_fast).mean()
    combined_df['Slow_MA'] = combined_df['Prices'].rolling(window = best_slow).mean()
    combined_df['Return'] = combined_df['Prices'].pct_change()
    combined_df['Signal'] = np.where(combined_df['Fast_MA'] >  combined_df['Slow_MA'],1,0)
    combined_df['Strategy_Return']= combined_df['Signal'].shift(1) * combined_df['Return']
    test_df = combined_df.iloc[train_end:train_end+5].copy()
    sharpe_test = test_df['Strategy_Return'].mean()/test_df['Strategy_Return'].std()
    test_result.append([train_end,best_fast,best_slow,sharpe_test])
    train_end+=5
    

final = pd.DataFrame(test_result,columns=['Train_End','Fast','Slow','Sharpe'])
final.sort_values(by='Sharpe',ascending = False,inplace = True)

print(final)

avg_sharpe = final['Sharpe'].mean()
print('Average Sharpe is : ', avg_sharpe)

# Here We Have Used Walk Forward Testing. Here We Tested whether our strategy that worked with Trained trained also works with unseen data.
# If It Works If Sharpe is positive on Seen and Unseen data for a particular scenario it means It's good to go else not.




                

    

   Train_End  Fast  Slow    Sharpe
0         20     5     6  0.662290
1         25     5     6  0.549489
2         30     5     6  0.302864
Average Sharpe is :  0.5048809467348574


In [4]:
# Proect 9 Trade Count + Parameter Stability Analysis
# Stability is tested by whether in all scenarios , Whether best MAs are same or different

import pandas as pd
import numpy as np

prices = [
    100, 102, 104, 101, 98,
    105, 110, 108, 112, 115,
    109, 107, 111, 117, 120,
    118, 121, 125, 123, 128,
    130, 127, 132, 135, 138,
    136, 140, 145, 143, 148,
    150, 147, 152, 155, 153,
    158, 160, 157, 162, 165
]

df = pd.DataFrame({'Prices': prices})
test_result = []
train_end = 20

for i in range(0,3):
    result = []
    train_df = df.iloc[:train_end].copy()
    
    for fast in range(2,6):
        for slow in range(6,11):
            trade_count = 0
            train_df['Fast_MA'] = train_df['Prices'].rolling(window=fast).mean()
            train_df['Slow_MA'] = train_df['Prices'].rolling(window=slow).mean()
            train_df['Return'] = train_df['Prices'].pct_change()
            train_df['Signal'] = np.where(train_df['Fast_MA'] > train_df['Slow_MA'] ,1,0 )
            train_df['Trade']= np.where(train_df['Signal'] != train_df['Signal'].shift(1),1,0)
            trade_count= train_df['Trade'].sum()
            train_df['Strategy_Return'] = train_df['Signal'].shift(1) * train_df['Return']
            sharpe= train_df['Strategy_Return'].mean()/train_df['Strategy_Return'].std()
            result.append([fast,slow,sharpe,trade_count])
    
    train_df1 = pd.DataFrame(result,columns=['Fast','Slow','Sharpe','Trade Count'])
    train_df1.sort_values(by='Sharpe',ascending = False,inplace = True)
    #print(train_df1.sort_values(by='Trade Count',ascending = False))
    best_fast = train_df1['Fast'].iloc[0]
    best_slow = train_df1['Slow'].iloc[0]
    combined_df = df.iloc[:train_end+5].copy()
    combined_df['Fast_MA'] = combined_df['Prices'].rolling(window = best_fast).mean()
    combined_df['Slow_MA'] = combined_df['Prices'].rolling(window = best_slow).mean()
    combined_df['Return'] = combined_df['Prices'].pct_change()
    combined_df['Signal'] = np.where(combined_df['Fast_MA'] >  combined_df['Slow_MA'],1,0)
    combined_df['Strategy_Return']= combined_df['Signal'].shift(1) * combined_df['Return']
    test_df = combined_df.iloc[train_end:train_end+5].copy()
    print(test_df,'test_df')
    sharpe_test = test_df['Strategy_Return'].mean()/test_df['Strategy_Return'].std()
    test_result.append([train_end,best_fast,best_slow,sharpe_test])
    train_end+=5

final = pd.DataFrame(test_result,columns=['Train_End','Fast','Slow','Sharpe'])
final.sort_values(by='Sharpe',ascending = False,inplace = True)
print(final[['Fast','Slow']].value_counts())

#print(final)

avg_sharpe = final['Sharpe'].mean()
#print('Average Sharpe is : ', avg_sharpe)

# Observation:
# Trade Count: As per backtest it is observed that MAs with lower difference have more trades
# Sharpe : With Forward Testing Sharpe Value kept on Decreasing. But still it was enough to make it a Good Strategy
# Strability : The strategy is fully Stable as in all forward test best Slow And Fast MA was same.




                

    

    Prices  Fast_MA     Slow_MA    Return  Signal  Strategy_Return
20     130    125.4  124.166667  0.015625       1         0.015625
21     127    126.6  125.666667 -0.023077       1        -0.023077
22     132    128.0  127.500000  0.039370       1         0.039370
23     135    130.4  129.166667  0.022727       1         0.022727
24     138    132.4  131.666667  0.022222       1         0.022222 test_df
    Prices  Fast_MA     Slow_MA    Return  Signal  Strategy_Return
25     136    133.6  133.000000 -0.014493       1        -0.014493
26     140    136.2  134.666667  0.029412       1         0.029412
27     145    138.8  137.666667  0.035714       1         0.035714
28     143    140.4  139.500000 -0.013793       1        -0.013793
29     148    142.4  141.666667  0.034965       1         0.034965 test_df
    Prices  Fast_MA     Slow_MA    Return  Signal  Strategy_Return
30     150    145.2  143.666667  0.013514       1         0.013514
31     147    146.6  145.500000 -0.020000     

,Prices,Fast_MA,Slow_MA,Return,Signal,Strategy_Return
30,150,145.2,143.666667,0.013514,1,0.013514
31,147,146.6,145.500000,-0.020000,1,-0.020000
32,152,148.0,147.500000,0.034014,1,0.034014
33,155,150.4,149.166667,0.019737,1,0.019737
34,153,151.4,150.833333,-0.012903,1,-0.012903


In [30]:
# Proect 10 Monte Carlo Stress Testing
# In this we check whether strategy works if strategy returns are reorderd.

returns = test_df['Strategy_Return'].dropna().values

simulation_result = []
for i in range(1,100):
    random_returns = np.random.permutation(returns)
    equity_curve = (1 + random_returns).cumprod()
    running_max = np.maximum.accumulate(equity_curve)
    drawdown = (equity_curve - running_max)/running_max
    max_drawdown = drawdown.min()
    simulation_result.append([i ,equity_curve[-1],max_drawdown])

#print(simulation_result)
df = pd.DataFrame(simulation_result,columns=['Simulation','Equity_Curve','Max_Drawdown'])
avg_maxdrawdown = df['Max_Drawdown'].mean()
worst_maxdrawdown = df['Max_Drawdown'].min()
best_maxdrawdown = df['Max_Drawdown'].max()

print('Avg Max Drwadown is :',avg_maxdrawdown)
print('Worst Drawdown is : ',worst_maxdrawdown)
print('Best Drawdown is : ',best_maxdrawdown)


Avg Max Drwadown is : -0.022024112088628242
Worst Drawdown is :  -0.032645161290322716
Best Drawdown is :  -0.012903225806451592
